# 13 — LightGBM physics: reproducible submission candidate

**Group 119 — Henrik Remmen (541747)**  
**Kaggle team:** `[119] SINTEF Databreach`

Train one binary LightGBM model per generator using the physical feature set. The fixed tree counts come from the completed three-fold physics LightGBM experiment that produced the uploaded CSV, so its final model can be reproduced without repeating validation.

This is a **self-contained, submission-ready candidate**, not a validation notebook. It starts from the supplied raw data, works offline, imports no project source file and needs no pretrained artifact. Run all cells from top to bottom. The output is `submissions/lgbm_physics_v2.csv`.

Course-hardware constraints are enforced: CPU only, at most four threads, sequential model fitting and float32 feature matrices. Completed task predictions are checkpointed, so an interrupted run can resume. The final selected short notebooks must still be benchmarked from a clean directory before submission; only two candidates will eventually be handed in.

## 1. Configuration and enforced hardware limit

In [ ]:
from pathlib import Path
import os
from IPython.display import display

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data/kernel").exists() and (path / "prediction_mapping.csv").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project containing data/ and prediction_mapping.csv."
    )

# The grading note defines a normal PC as at most 4 CPU cores and 32 GB RAM.
CPU_BUDGET = min(4, max(1, os.cpu_count() or 1))
THREADS_PER_MODEL = CPU_BUDGET
for key in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ["OMP_NUM_THREADS"] = str(THREADS_PER_MODEL)

CANDIDATE_NAME = 'lgbm_physics_v2'
BACKEND = 'lightgbm'
MODE = 'unit'
FINAL_ROUNDS = {'Hogga_G1': 398, 'Tokke_G1': 534, 'Tokke_G2': 387, 'Tokke_G3': 458, 'Tokke_G4': 410, 'Lio_G1': 386, 'Byrte_G1': 441, 'Vinje_G1': 852, 'Vinje_G2': 299, 'Vinje_G3': 873, 'Haukeli_G1': 330, 'Songa_G1': 465, 'Kjela_G1': 393, 'Vesle Kjela_G1': 177}

print("Root:", ROOT)
print(f"CPU limit: {THREADS_PER_MODEL} thread(s); models run sequentially")
print("Training has not started.")

## 2. Embedded raw-data and feature pipeline

In [ ]:
"""Shared, target-safe data and feature pipeline used by experiment notebooks.

The module contains no model training.  It exists so CatBoost and XGBoost
experiments use exactly the same inputs, folds, metric, and submission layout.
"""

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score


HORIZON = 168
RANDOM_SEED = 42


@dataclass
class ProjectData:
    root: Path
    X_all: pd.DataFrame
    target_cube: np.ndarray
    units: list[str]
    train_cases: pd.DataFrame
    test_cases: pd.DataFrame
    sample_submission: pd.DataFrame
    feature_sets: dict[str, list[str]]
    feature_group: dict[str, str]
    generator_features: pd.DataFrame
    folds: list[dict]

    @property
    def n_train_cases(self):
        return len(self.train_cases)

    @property
    def n_test_cases(self):
        return len(self.test_cases)

    @property
    def n_all_cases(self):
        return self.n_train_cases + self.n_test_cases


def read_time_csv(path, time_column="date"):
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column], utc=True)
    return frame


def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None] * HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()


def micro_auc(y_true_cube, y_pred_cube):
    if y_true_cube.shape != y_pred_cube.shape:
        raise ValueError("Truth and prediction cubes must have identical shapes")
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )


def per_unit_auc(y_true_cube, y_pred_cube, units):
    return {
        unit: roc_auc_score(
            y_true_cube[:, unit_index, :].ravel(),
            y_pred_cube[:, unit_index, :].ravel(),
        )
        for unit_index, unit in enumerate(units)
    }


def make_purged_folds(train_cases, validation_years=(2020, 2021, 2022)):
    starts = pd.DatetimeIndex(train_cases["starttime"])
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
        validation_cases = np.flatnonzero(
            (starts >= validation_start) & (starts < validation_end)
        )
        training_cases = np.flatnonzero(
            starts + pd.Timedelta(hours=HORIZON - 1) < validation_start
        )
        if not len(training_cases) or not len(validation_cases):
            raise ValueError(f"Empty training or validation partition for {year}")
        if not (
            starts[training_cases].max() + pd.Timedelta(hours=HORIZON - 1)
            < starts[validation_cases].min()
        ):
            raise AssertionError(f"Fold {year} is not temporally purged")
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds


def make_submission(data, prediction_cube, filename):
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if prediction_cube.shape != expected:
        raise ValueError(f"Expected predictions {expected}, got {prediction_cube.shape}")

    submission_data = {"Run No": data.test_cases["Run No"].to_numpy()}
    for unit_index, unit in enumerate(data.units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_index, horizon]
            )

    submission = pd.DataFrame(submission_data)[data.sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise AssertionError("Run No does not match sample_submission.csv")
    if not np.isfinite(probabilities).all():
        raise ValueError("Submission contains missing or non-finite predictions")
    if probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError("Submission probabilities are outside [0, 1]")

    path = data.root / "submissions" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path, submission


def build_project_data(root):
    root = Path(root).resolve()
    data_dir = root / "data"

    price = read_time_csv(data_dir / "kernel" / "Historical_day_ahead_price_2015_2025.csv")
    inflow = read_time_csv(data_dir / "kernel" / "Historical_inflow_1958_2025.csv")
    volume = read_time_csv(data_dir / "kernel" / "Historical_volume_2015_2024.csv")
    water_value = read_time_csv(data_dir / "kernel" / "Synthetic_water_value_2015_2024.csv")
    min_flow = read_time_csv(data_dir / "extended" / "Constraint_min_flow.csv")
    min_volume = read_time_csv(data_dir / "extended" / "Constraint_min_volume.csv")
    targets = read_time_csv(data_dir / "kernel" / "Unit_commitment_decisions.csv", "starttime")
    prediction_mapping = read_time_csv(root / "prediction_mapping.csv", "starttime")
    sample_submission = pd.read_csv(root / "sample_submission.csv")

    with open(data_dir / "extended" / "Tokke_Vinje_topology.yaml") as stream:
        topology = yaml.safe_load(stream)

    pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
    target_columns = [column for column in targets if pattern.match(column)]
    units = list(dict.fromkeys(pattern.match(column).group(1) for column in target_columns))
    if len(units) != 14 or len(target_columns) != 14 * HORIZON:
        raise AssertionError("Expected 14 units and 168 targets per unit")
    if target_columns != list(sample_submission.columns[1:]):
        raise AssertionError("Target order differs from sample submission")

    target_cube = np.stack([
        targets[[f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]]
        .to_numpy(dtype=np.int8)
        for unit in units
    ], axis=1)
    if not set(np.unique(target_cube)).issubset({0, 1}):
        raise AssertionError("Targets must be binary")

    train_cases = targets[["Run No", "starttime"]].copy()
    test_cases = prediction_mapping[["Run No", "starttime"]].copy()
    all_cases = pd.concat([train_cases, test_cases], ignore_index=True)
    n_all_cases = len(all_cases)
    starts = pd.DatetimeIndex(all_cases["starttime"])
    horizon_hour = np.tile(np.arange(HORIZON), n_all_cases)
    target_time = pd.DatetimeIndex(
        starts.repeat(HORIZON) + pd.to_timedelta(horizon_hour, unit="h")
    )

    def numeric_indexed(frame):
        return frame.set_index("date").select_dtypes(include=np.number)

    def hourly_matrix(series):
        values = series.reindex(target_time).to_numpy()
        matrix = values.reshape(n_all_cases, HORIZON)
        if not np.isfinite(matrix).all():
            raise ValueError(
                f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
            )
        return matrix

    def repeat_case(values):
        values = np.asarray(values)
        if len(values) != n_all_cases:
            raise ValueError("Case-level array has incorrect length")
        return np.repeat(values, HORIZON)

    def cyclic(values, period):
        angle = 2 * np.pi * np.asarray(values) / period
        return np.sin(angle), np.cos(angle)

    def clipped_shift(matrix, offset):
        indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON - 1)
        return matrix[:, indices]

    def centred_rolling_mean(matrix, window):
        return (
            pd.DataFrame(matrix).T
            .rolling(window=window, min_periods=1, center=True)
            .mean().T.to_numpy()
        )

    feature_data = {}
    feature_group = {}

    def add_feature(name, values, group):
        values = np.asarray(values)
        expected = n_all_cases * HORIZON
        if values.ndim != 1 or len(values) != expected:
            raise ValueError(f"{name}: expected {expected}, got {values.shape}")
        feature_data[name] = values.astype(np.float32)
        feature_group[name] = group

    hour = target_time.hour.to_numpy()
    weekday = target_time.dayofweek.to_numpy()
    month = target_time.month.to_numpy()
    for name, values, period in (
        ("hour", hour, 24),
        ("weekday", weekday, 7),
        ("month", month - 1, 12),
    ):
        sine, cosine = cyclic(values, period)
        add_feature(f"{name}_sin", sine, "calendar")
        add_feature(f"{name}_cos", cosine, "calendar")
    add_feature("horizon_hour", horizon_hour, "calendar")

    price_matrix = hourly_matrix(
        price.set_index("date")["Day-ahead price (EUR/MWh)"]
    )
    price_mean = price_matrix.mean(axis=1)
    price_std = price_matrix.std(axis=1)
    price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()
    add_feature("price", price_matrix.ravel(), "price")
    add_feature("price_week_mean", repeat_case(price_mean), "price")
    add_feature("price_week_std", repeat_case(price_std), "price")
    add_feature("price_week_percentile", price_rank.ravel(), "price")
    add_feature("price_minus_week_mean", (price_matrix - price_mean[:, None]).ravel(), "price")
    add_feature(
        "price_week_zscore",
        ((price_matrix - price_mean[:, None]) / (price_std[:, None] + 1e-6)).ravel(),
        "price_windows",
    )
    add_feature("price_top_10pct", (price_rank >= 0.90).ravel(), "price_windows")
    add_feature("price_bottom_10pct", (price_rank <= 0.10).ravel(), "price_windows")
    for offset in (1, 3, 6, 24):
        lag = clipped_shift(price_matrix, -offset)
        lead = clipped_shift(price_matrix, offset)
        add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
        add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
        add_feature(f"price_change_from_{offset}h", (price_matrix - lag).ravel(), "price_windows")
        add_feature(f"price_change_to_next_{offset}h", (lead - price_matrix).ravel(), "price_windows")
    for window in (3, 6, 12, 24, 48):
        rolling = centred_rolling_mean(price_matrix, window)
        add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
        add_feature(f"price_minus_mean_{window}h", (price_matrix - rolling).ravel(), "price_windows")

    reservoirs = [column for column in volume.columns if column != "date"]
    capacity = {
        name: values["max_vol"]
        for name, values in topology["model"]["reservoir"].items()
    }
    start_volume = numeric_indexed(volume).reindex(starts)[reservoirs]
    start_water = numeric_indexed(water_value).reindex(starts)[reservoirs]
    end_water = numeric_indexed(water_value).reindex(
        starts + pd.Timedelta(days=7)
    )[reservoirs]
    if start_volume.isna().any().any() or start_water.isna().any().any() or end_water.isna().any().any():
        raise ValueError("Missing case-level reservoir data")

    for reservoir in reservoirs:
        fill = start_volume[reservoir].to_numpy() / capacity[reservoir]
        water_start = start_water[reservoir].to_numpy()
        water_end = end_water[reservoir].to_numpy()
        add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
        add_feature(f"water_value_end__{reservoir}", repeat_case(water_end), "water_margins")
        add_feature(f"water_value_change__{reservoir}", repeat_case(water_end - water_start), "water_margins")
        add_feature(f"price_margin__{reservoir}", (price_matrix - water_end[:, None]).ravel(), "water_margins")

    system_fill = (
        start_volume.sum(axis=1).to_numpy()
        / sum(capacity[reservoir] for reservoir in reservoirs)
    )
    water_end_mean = end_water.mean(axis=1).to_numpy()
    add_feature("system_filling_ratio", repeat_case(system_fill), "hydrology")
    add_feature("water_value_end_mean", repeat_case(water_end_mean), "water_margins")
    margin = price_matrix - water_end_mean[:, None]
    add_feature("price_minus_water_value_mean", margin.ravel(), "water_margins")
    add_feature(
        "price_to_abs_water_value_ratio",
        (price_matrix / (np.abs(water_end_mean[:, None]) + 1.0)).ravel(),
        "water_margins",
    )

    inflow_values = numeric_indexed(inflow)
    inflow_matrices = {}
    for column in inflow_values:
        matrix = hourly_matrix(inflow_values[column])
        inflow_matrices[column] = matrix
        add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
        add_feature(f"inflow_week_mean__{column}", repeat_case(matrix.mean(axis=1)), "hydrology")
    total_inflow = np.sum(list(inflow_matrices.values()), axis=0)
    add_feature("inflow_total", total_inflow.ravel(), "hydrology")
    add_feature("inflow_total_week_mean", repeat_case(total_inflow.mean(axis=1)), "hydrology")
    add_feature("inflow_total_week_std", repeat_case(total_inflow.std(axis=1)), "hydrology")
    add_feature("inflow_total_24h_mean", centred_rolling_mean(total_inflow, 24).ravel(), "hydrology")
    add_feature(
        "inflow_total_cumulative_mean",
        (np.cumsum(total_inflow, axis=1) / (np.arange(HORIZON)[None, :] + 1)).ravel(),
        "hydrology",
    )

    flow_values = numeric_indexed(min_flow)
    min_volume_values = numeric_indexed(min_volume)
    flow_matrices = {}
    min_volume_matrices = {}
    for column in flow_values:
        matrix = hourly_matrix(flow_values[column])
        flow_matrices[column] = matrix
        add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")
    for column in min_volume_values:
        matrix = hourly_matrix(min_volume_values[column])
        min_volume_matrices[column] = matrix
        add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
        if column in reservoirs:
            initial = start_volume[column].to_numpy()[:, None]
            add_feature(
                f"min_volume_slack__{column}",
                ((initial - matrix) / capacity[column]).ravel(),
                "constraints",
            )

    flow_active_count = np.sum([matrix > 0 for matrix in flow_matrices.values()], axis=0)
    volume_active_count = np.sum([matrix > 0 for matrix in min_volume_matrices.values()], axis=0)
    total_min_flow = np.sum(list(flow_matrices.values()), axis=0)
    total_min_volume = np.sum(list(min_volume_matrices.values()), axis=0)
    add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
    add_feature("min_volume_active_count", volume_active_count.ravel(), "constraints")
    add_feature("min_flow_total", total_min_flow.ravel(), "constraints")
    add_feature("min_volume_total", total_min_volume.ravel(), "constraints")

    variable_flow_columns = [
        column for column in flow_values
        if 0 < (flow_values[column] > 0).mean() < 1
    ]
    variable_volume_columns = [
        column for column in min_volume_values
        if 0 < (min_volume_values[column] > 0).mean() < 1
    ]
    variable_active = np.zeros((n_all_cases, HORIZON), dtype=bool)
    for column in variable_flow_columns:
        variable_active |= flow_matrices[column] > 0
    for column in variable_volume_columns:
        variable_active |= min_volume_matrices[column] > 0
    add_feature("variable_constraint_active", variable_active.ravel(), "constraints")

    add_feature("margin_x_system_fill", (margin * system_fill[:, None]).ravel(), "interactions")
    add_feature("price_rank_x_system_fill", (price_rank * system_fill[:, None]).ravel(), "interactions")
    add_feature("margin_x_total_inflow", (margin * total_inflow).ravel(), "interactions")
    add_feature("price_rank_x_hour_sin", price_rank.ravel() * feature_data["hour_sin"], "interactions")
    add_feature("price_rank_x_hour_cos", price_rank.ravel() * feature_data["hour_cos"], "interactions")
    add_feature("margin_x_constraint", margin.ravel() * variable_active.ravel(), "interactions")

    X_all = pd.DataFrame(feature_data)
    constants = [
        column for column in X_all
        if X_all[column].nunique(dropna=False) <= 1
    ]
    X_all = X_all.drop(columns=constants)
    if X_all.isna().any().any() or not np.isfinite(X_all.to_numpy()).all():
        raise ValueError("Final feature matrix contains invalid values")

    baseline = [
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
        "month_sin", "month_cos", "horizon_hour", "price",
        "price_week_mean", "price_week_std", "price_week_percentile",
        "price_minus_week_mean", "water_value_end_mean",
        "price_minus_water_value_mean", "system_filling_ratio",
        "inflow_total", "inflow_total_24h_mean", "min_flow_total",
        "min_volume_total", "variable_constraint_active",
    ]

    def grouped(name):
        return [
            feature for feature, group in feature_group.items()
            if group == name and feature in X_all.columns
        ]

    def unique_existing(names):
        return list(dict.fromkeys(name for name in names if name in X_all.columns))

    feature_sets = {
        "baseline": unique_existing(baseline),
        "baseline_plus_price": unique_existing(baseline + grouped("price_windows")),
        "baseline_plus_price_water": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
        ),
        "baseline_plus_price_water_hydrology": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology")
        ),
        "full_engineered": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology") + grouped("constraints")
            + grouped("interactions")
        ),
    }

    generator_rows = []
    generator_config = topology["model"]["generator"]
    for unit in units:
        config = generator_config[unit]
        plant = unit.rsplit("_G", 1)[0]
        number_match = re.search(r"_G(\d+)$", unit)
        generator_rows.append({
            "unit": unit,
            "plant": plant,
            "generator_number": int(number_match.group(1)) if number_match else 1,
            "p_min": config.get("p_min", np.nan),
            "p_max": config.get("p_max", np.nan),
            "p_nom": config.get("p_nom", np.nan),
            "minimum_output_fraction": config.get("p_min", np.nan) / config.get("p_max", np.nan),
            "start_cost": config.get("startcost_const", 0.0),
            "stop_cost": config.get("stopcost_const", 0.0),
            "penstock": str(config.get("penstock", "unknown")),
        })
    generator_features = pd.DataFrame(generator_rows)
    generator_features["units_in_plant"] = (
        generator_features.groupby("plant")["unit"].transform("count")
    )

    return ProjectData(
        root=root,
        X_all=X_all,
        target_cube=target_cube,
        units=units,
        train_cases=train_cases,
        test_cases=test_cases,
        sample_submission=sample_submission,
        feature_sets=feature_sets,
        feature_group=feature_group,
        generator_features=generator_features,
        folds=make_purged_folds(train_cases),
    )


## 3. Embedded physical feature definitions

In [ ]:
"""Deterministic input-only features for the next hydropower experiments.

No fitted transforms, labels, predicted/observed future storage, or solver outputs.
Water budgets deliberately exclude unknown dispatch and are explicitly proxies.
"""

import numpy as np
import pandas as pd
import yaml



def window_sum(values, width, forward=True):
    """Truncated within-case sum, including the current hour; no edge repeats."""
    if not forward:
        return window_sum(values[:, ::-1], width, True)[:, ::-1]
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    left = np.arange(values.shape[1])
    right = np.minimum(left + width, values.shape[1])
    return cumulative[:, right] - cumulative[:, left]


def consecutive_true(mask):
    """Run length ending/starting at each position, reset at each case boundary."""
    past = np.zeros(mask.shape, dtype=np.float32)
    future = np.zeros_like(past)
    for h in range(mask.shape[1]):
        past[:, h] = mask[:, h] * (1 + (past[:, h - 1] if h else 0))
    for h in range(mask.shape[1] - 1, -1, -1):
        future[:, h] = mask[:, h] * (1 + (future[:, h + 1] if h + 1 < mask.shape[1] else 0))
    return past, future


def plant_sources(topology, plant):
    """Follow reservoir -> plant or reservoir -> tunnel -> plant connections.

    Stop at the first reservoir; do not pretend all upstream water arrives instantly.
    """
    connections = topology['connections']
    found = set()

    def visit(name, kind, seen):
        if (name, kind) in seen:
            return
        seen = seen | {(name, kind)}
        for edge in connections:
            if edge['to'] != name or edge['to_type'] != kind:
                continue
            if edge['from_type'] == 'reservoir':
                found.add(edge['from'])
            elif edge['from_type'] == 'tunnel':
                visit(edge['from'], 'tunnel', seen)

    visit(plant, 'plant', set())
    if not found:
        raise ValueError(f'No source reservoir found for {plant}')
    return sorted(found)


def build_research_features(data):
    """Return global additions, per-plant additions, and an auditable source mapping."""
    root = data.root
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    n = data.n_all_cases
    horizon = np.arange(HORIZON)
    time_index = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(horizon, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(time_index)
    minimum = read_time_csv(root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(time_index)
    extra = {}

    def add(target, name, values):
        values = np.broadcast_to(values, (n, HORIZON)).reshape(-1).astype(np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite research feature {name}')
        target[name] = values

    add(extra, 'v2_day_in_horizon', np.broadcast_to(horizon // 24, (n, HORIZON)))
    add(extra, 'v2_hours_remaining', HORIZON - horizon)
    day_price = price.reshape(n, 7, 24)
    day_mean = np.repeat(day_price.mean(axis=2), 24, axis=1)
    day_rank = pd.DataFrame(day_price.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy().reshape(n, HORIZON)
    add(extra, 'v2_price_day_rank', day_rank)
    add(extra, 'v2_price_minus_day_mean', price - day_mean)
    prefix_mean = np.cumsum(price, axis=1) / (horizon + 1)
    suffix_mean = np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - horizon)
    add(extra, 'v2_price_minus_prefix_mean', price - prefix_mean)
    add(extra, 'v2_price_minus_suffix_mean', price - suffix_mean)
    add(extra, 'v2_best_remaining_price_gap', np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price)
    reservoirs = topology['model']['reservoir']
    reservoir_inputs = {}
    for name, config in reservoirs.items():
        cap = float(config['max_vol'])
        initial = volume[name].to_numpy()[:, None]
        terminal = water[name].to_numpy()[:, None]
        flow = inflow[name].to_numpy().reshape(n, HORIZON)
        # m3/s * 3600 / 1e6 -> million m3 per hour, matching storage units.
        hourly_volume = flow * 0.0036
        cumulative = np.cumsum(hourly_volume, axis=1)
        remaining = np.cumsum(hourly_volume[:, ::-1], axis=1)[:, ::-1]
        floor = minimum[name].to_numpy().reshape(n, HORIZON) if name in minimum else np.zeros_like(price)
        budget = initial + cumulative
        add(extra, f'v2_cumulative_inflow_fill__{name}', cumulative / cap)
        add(extra, f'v2_remaining_inflow_fill__{name}', remaining / cap)
        add(extra, f'v2_no_release_excess__{name}', np.maximum(budget - cap, 0) / cap)
        add(extra, f'v2_no_release_min_slack__{name}', (budget - floor) / cap)
        curve = config['vol_head']
        head = np.interp(initial[:, 0], curve['x'], curve['y'])[:, None]
        reservoir_inputs[name] = (cap, initial, terminal, cumulative, remaining, head)

    plants = {}
    mappings = []
    for plant, plant_config in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        values = [reservoir_inputs[s] for s in sources]
        capacity = sum(v[0] for v in values)
        initial = sum(v[1] for v in values)
        # Multi-reservoir summaries are capacity-weighted approximations, not SHOP prices.
        terminal = sum(v[2] * v[0] for v in values) / capacity
        head = sum(v[5] * v[0] for v in values) / capacity - float(plant_config['outlet_line'])
        cum = sum(v[3] for v in values)
        remain = sum(v[4] for v in values)
        margin = price - terminal
        out = {}
        add(out, 'local_initial_fill', initial / capacity)
        add(out, 'local_gross_head_proxy', head)
        add(out, 'local_terminal_water_value', terminal)
        add(out, 'local_margin', margin)
        add(out, 'local_margin_normalized', margin / (np.abs(terminal) + price.std(axis=1)[:, None] + 1))
        add(out, 'local_cumulative_inflow_fill', cum / capacity)
        add(out, 'local_remaining_inflow_fill', remain / capacity)
        add(out, 'local_margin_x_fill', margin * initial / capacity)
        positive = np.maximum(margin, 0)
        negative = np.maximum(-margin, 0)
        past, future = consecutive_true(margin > 0)
        add(out, 'local_profitable_run_past', past)
        add(out, 'local_profitable_run_future', future)
        add(out, 'local_profitable_fraction_week', (margin > 0).mean(axis=1)[:, None])
        for hours in (6, 24):
            add(out, f'local_positive_margin_next_{hours}h', window_sum(positive, hours))
            add(out, f'local_negative_margin_next_{hours}h', window_sum(negative, hours))
            add(out, f'local_positive_margin_past_{hours}h', window_sum(positive, hours, False))
        # Start cost / rated production gives EUR/MWh; margin sums are hourly
        # EUR/MWh contributions. This is an approximate rated-power payback ratio.
        unit_configs = [topology['model']['generator'][u] for u in data.units if u.rsplit('_G', 1)[0] == plant]
        cost_per_mw = np.mean([(v.get('startcost_const', 0) + v.get('stopcost_const', 0)) / v['p_max'] for v in unit_configs])
        add(out, 'local_start_stop_payback_proxy', window_sum(positive, 24) / max(cost_per_mw, 1e-6))
        plants[plant] = pd.DataFrame(out)
        mappings.append({'plant': plant, 'source_reservoirs': '|'.join(sources), 'source_capacity_Mm3': capacity})
    return pd.DataFrame(extra), plants, pd.DataFrame(mappings)


## 4. Fixed-model reproduction helpers

In [ ]:
import gc
import json
import time


def experiment_tasks(data, mode):
    if mode == "unit":
        return [(unit, unit.rsplit("_G", 1)[0], [index])
                for index, unit in enumerate(data.units)]
    plants = list(dict.fromkeys(unit.rsplit("_G", 1)[0] for unit in data.units))
    return [(plant, plant,
             [index for index, unit in enumerate(data.units)
              if unit.rsplit("_G", 1)[0] == plant])
            for plant in plants]


def encode_states(cube):
    """case x unit x hour -> case-hour integer ON/OFF state."""
    bits = 1 << np.arange(cube.shape[1])
    return (cube.transpose(0, 2, 1).reshape(-1, cube.shape[1]) @ bits).astype(np.int32)


def state_marginals(probabilities, n_units):
    probabilities = np.asarray(probabilities)
    if n_units == 1:
        return probabilities.reshape(-1, 1)
    states = np.arange(1 << n_units)
    bits = ((states[:, None] >> np.arange(n_units)) & 1).astype(np.float64)
    return probabilities @ bits


def fit_fixed_model(backend, X, y, n_units, rounds):
    if backend == "xgboost":
        from xgboost import XGBClassifier
        if n_units != 1:
            raise ValueError("XGBoost candidate expects one binary model per generator")
        model = XGBClassifier(
            objective="binary:logistic", eval_metric="auc", n_estimators=int(rounds),
            learning_rate=0.03, max_depth=7, min_child_weight=50,
            subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1,
            reg_lambda=5.0, gamma=0.0, tree_method="hist", max_bin=256,
            device="cpu", n_jobs=THREADS_PER_MODEL, random_state=RANDOM_SEED,
        )
        model.fit(X, y, verbose=False)
        return model

    import lightgbm as lgb
    parameters = dict(
        objective="binary" if n_units == 1 else "multiclass",
        metric="None", learning_rate=0.03, num_leaves=31, max_depth=-1,
        min_data_in_leaf=150, feature_fraction=0.85,
        bagging_fraction=0.85, bagging_freq=1, lambda_l1=0.1,
        lambda_l2=5.0, max_bin=255, num_threads=THREADS_PER_MODEL,
        device_type="cpu", deterministic=True, force_col_wise=True,
        seed=RANDOM_SEED, verbosity=-1,
    )
    if n_units > 1:
        parameters["num_class"] = 1 << n_units
    dataset = lgb.Dataset(X, label=y, free_raw_data=True)
    return lgb.train(parameters, dataset, num_boost_round=int(rounds))


def predict_model(model, backend, X, n_units):
    if backend == "xgboost":
        return model.predict_proba(X)[:, 1, None]
    return state_marginals(model.predict(X), n_units)


class SubmissionRunner:
    """Raw data -> fixed models -> Kaggle CSV, using at most four CPU threads."""

    def __init__(self, data, extra, local, name, backend, mode, rounds):
        self.data = data
        self.local = local
        self.name = name
        self.backend = backend
        self.mode = mode
        self.rounds = dict(rounds)
        self.tasks = experiment_tasks(data, mode)
        task_names = [task[0] for task in self.tasks]
        if set(task_names) != set(self.rounds):
            raise ValueError(f"Round constants do not match tasks: {task_names}")

        base_columns = data.feature_sets["full_engineered"]
        self.base = np.column_stack([
            data.X_all[base_columns].to_numpy(dtype=np.float32),
            extra.to_numpy(dtype=np.float32),
        ])
        self.base = np.ascontiguousarray(self.base, dtype=np.float32)
        if not np.isfinite(self.base).all():
            raise ValueError("Global feature matrix contains non-finite values")

        self.checkpoint_dir = data.root / "artifacts" / "reproduction" / name
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def matrix(self, plant):
        matrix = np.column_stack([
            self.base,
            self.local[plant].to_numpy(dtype=np.float32),
        ])
        matrix = np.ascontiguousarray(matrix, dtype=np.float32)
        if not np.isfinite(matrix).all():
            raise ValueError(f"Feature matrix for {plant} contains non-finite values")
        return matrix

    def labels(self, case_ids, unit_ids):
        cube = self.data.target_cube[np.asarray(case_ids)][:, unit_ids, :]
        if self.mode == "unit":
            return cube.reshape(-1).astype(np.int32)
        return encode_states(cube)

    def run(self):
        train_cases = np.arange(self.data.n_train_cases)
        test_cases = np.arange(self.data.n_train_cases, self.data.n_all_cases)
        train_rows = rows_for_cases(train_cases)
        test_rows = rows_for_cases(test_cases)
        prediction_cube = np.empty(
            (self.data.n_test_cases, len(self.data.units), HORIZON),
            dtype=np.float32,
        )
        timing_rows = []
        total_start = time.perf_counter()

        for task_index, (task_name, plant, unit_ids) in enumerate(self.tasks):
            checkpoint = self.checkpoint_dir / f"task_{task_index:02d}.npz"
            started = time.perf_counter()
            if checkpoint.exists():
                with np.load(checkpoint, allow_pickle=False) as saved:
                    if (saved["task_name"].item() != task_name
                            or int(saved["rounds"]) != self.rounds[task_name]
                            or not np.array_equal(saved["unit_ids"], unit_ids)):
                        raise ValueError(f"Stale checkpoint: {checkpoint}")
                    task_prediction = saved["prediction"].copy()
                status = "restored"
            else:
                matrix = self.matrix(plant)
                y_train = self.labels(train_cases, unit_ids)
                model = fit_fixed_model(
                    self.backend,
                    matrix[train_rows],
                    y_train,
                    len(unit_ids),
                    self.rounds[task_name],
                )
                raw = predict_model(model, self.backend, matrix[test_rows], len(unit_ids))
                task_prediction = raw.reshape(
                    self.data.n_test_cases, HORIZON, len(unit_ids)
                ).transpose(0, 2, 1).astype(np.float32)
                temporary = checkpoint.with_suffix(".tmp.npz")
                np.savez_compressed(
                    temporary,
                    task_name=np.asarray(task_name),
                    rounds=np.asarray(self.rounds[task_name]),
                    unit_ids=np.asarray(unit_ids),
                    prediction=task_prediction,
                )
                os.replace(temporary, checkpoint)
                status = "trained"
                del matrix, y_train, model, raw
                gc.collect()

            expected = (self.data.n_test_cases, len(unit_ids), HORIZON)
            if task_prediction.shape != expected or not np.isfinite(task_prediction).all():
                raise ValueError(f"Invalid prediction for {task_name}: {task_prediction.shape}")
            prediction_cube[:, unit_ids, :] = task_prediction
            seconds = time.perf_counter() - started
            timing_rows.append({
                "task": task_name,
                "rounds": self.rounds[task_name],
                "status": status,
                "seconds": seconds,
            })
            print(f"{task_name:18s} | {self.rounds[task_name]:4d} rounds | "
                  f"{status} | {seconds:.1f}s", flush=True)

        if prediction_cube.min() < 0 or prediction_cube.max() > 1:
            raise ValueError("Predictions are outside [0, 1]")
        submission_path, submission = make_submission(
            self.data, prediction_cube, f"{self.name}.csv"
        )
        timings = pd.DataFrame(timing_rows)
        timings.to_csv(self.checkpoint_dir / "runtime.csv", index=False)
        metadata = {
            "candidate": self.name,
            "backend": self.backend,
            "mode": self.mode,
            "cpu_threads": THREADS_PER_MODEL,
            "rounds": self.rounds,
            "elapsed_seconds": time.perf_counter() - total_start,
        }
        (self.checkpoint_dir / "run_metadata.json").write_text(
            json.dumps(metadata, indent=2), encoding="utf-8"
        )
        return submission_path, submission, timings, metadata

## 5. Build the dataset — no training

The 14 constants are the median inner-selected rounds from the completed lgbm_physics_v2 purged folds, exactly as used by the uploaded lgbm_physics_v2.csv. These counts were selected during development and are fixed for reproduction.

In [ ]:
data = build_project_data(ROOT)
extra_features, local_features, plant_mapping = build_research_features(data)
runner = SubmissionRunner(
    data=data,
    extra=extra_features,
    local=local_features,
    name=CANDIDATE_NAME,
    backend=BACKEND,
    mode=MODE,
    rounds=FINAL_ROUNDS,
)

print("Cases:", data.n_train_cases, "train /", data.n_test_cases, "test")
print("Tasks:", len(runner.tasks))
print("Global features:", runner.base.shape[1])
display(plant_mapping)

## 6. Training and submission

This cell performs only the fits required for the candidate CSV. It does **not** run cross-validation, a matched control, feature selection or hyperparameter search. Models run one at a time and use at most four threads in total.

In [ ]:
# This is the only training cell. It fits directly on all training history.
submission_path, submission, runtime_table, run_metadata = runner.run()

## 7. Output verification — no training

In [ ]:
assert submission_path.exists()
assert submission.columns.tolist() == data.sample_submission.columns.tolist()
assert submission["Run No"].equals(data.sample_submission["Run No"])
probabilities = submission.iloc[:, 1:].to_numpy()
assert probabilities.shape == (data.n_test_cases, len(data.units) * HORIZON)
assert np.isfinite(probabilities).all()
assert 0 <= probabilities.min() <= probabilities.max() <= 1

print("Saved:", submission_path)
print(f"Elapsed: {run_metadata['elapsed_seconds'] / 60:.1f} minutes")
display(runtime_table)
display(submission.iloc[:3, :8])